# Стоимость земли: воспроизводимый эксперимент

Внешний CatBoost не распространяется. `LAND_MODEL_PATH` / `PRIMORSK_LAND_MODEL` задаёт .cbm. Физические UTM-признаки — основной расчёт; альтернативная конвенция площади — чувствительность, не подтверждённая схема обучения. 101 признак, категории по CTR-хешам реальной модели; подробности — docs/land-model.md. Это не рыночная или кадастровая оценка.

In [1]:
from pathlib import Path
import csv, json, subprocess
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
assert (ROOT/'scripts/land_value.py').is_file()


In [2]:
result=subprocess.run([str(ROOT/'.venv-land/bin/python'),str(ROOT/'scripts/land_value.py')],cwd=ROOT,capture_output=True,text=True,check=True)
print(result.stdout)


{'scenario': 'today', 'blocks': 526, 'population': 13659, 'total_rub': 215233090727.7857, 'delta_rub': 0.0, 'unit_median_rub_m2': 1228.2446355728657}
{'scenario': 's1', 'blocks': 526, 'population': 13734, 'total_rub': 215236114254.21558, 'delta_rub': 3023526.429869402, 'unit_median_rub_m2': 1226.381530958303}
{'scenario': 's2', 'blocks': 526, 'population': 13734, 'total_rub': 215255467044.22815, 'delta_rub': 22376316.44244334, 'unit_median_rub_m2': 1226.381530958303}



In [3]:
passport=json.loads((ROOT/'reports/land_model_passport.json').read_text())
rows=list(csv.DictReader((ROOT/'reports/land_summary.csv').open()))
assert len(passport['model']['features'])==101
assert [int(r['population']) for r in rows]==[13659,13734,13734]
assert [int(r['blocks']) for r in rows]==[526,526,526]
assert float(rows[0]['delta_rub'])==0
for row in rows:
    assert float(row['total_rub'])>0 and float(row['unit_median_rub_m2'])>0
print('Проверено: 101 признак, стабильные 526 кварталов, население, нулевая дельта today.')
print('Ограничение:',passport['numeric_convention']['verification'])
print((ROOT/'reports/land_area_convention_sensitivity.csv').read_text())


Проверено: 101 признак, стабильные 526 кварталов, население, нулевая дельта today.
Ограничение: CBM numeric training area convention unverified: reference differs from CBM categories; no complete training provenance
scenario,physical_utm_total_rub,reference_webmercator_site_total_rub,sensitivity_minus_primary_rub,status
today,215233090727.7857,325459435700.41394,110226344972.62823,hypothesis_only_unverified_model_numeric_convention
s1,215236114254.21558,325499931976.3749,110263817722.1593,hypothesis_only_unverified_model_numeric_convention
s2,215255467044.22815,325548854000.41736,110293386956.18921,hypothesis_only_unverified_model_numeric_convention

